In [ ]:

import json
import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from datetime import datetime
import time
from collections import Counter
import pdfplumber
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import langid
import random
import spacy


class ClimateReportAnalyzer:
    # Chunking settings
    MIN_CHARS = 400
    MAX_CHARS = 2000
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 48

    # Translation settings
    TRANSLATE_BATCH_SIZE = 48
    TRANSLATE_INPUT_MAX_TOKENS = 400
    TRANSLATE_OUTPUT_MAX_TOKENS = 400

    def __init__(self, cache_dir="cache"):
        # ...

    def extract_pdf(self, pdf_path: str) -> Dict:
        """Extract text from PDF and chunk it intelligently"""
        # ...

def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract and preprocess text from PDF"""


    def _preprocess_text(self, text: str) -> str:
        """
        Enhanced text cleaning:
        1. Unicode normalization
        2. Remove hyphenation
        3. Filter noise (headers, footers, navigation)
        4. Reconstruct paragraphs
        """


    def _is_noise(self, line: str) -> bool:
        """Detect headers, footers, page numbers, navigation"""


    def _reconstruct_paragraphs(self, lines: List[str]) -> List[str]:
        """Reconstruct paragraphs, handling tables specially"""
        paragraphs = []



    def _is_table_row(self, line: str) -> bool:
        """Detect table rows (multiple numbers)"""



    def _should_break_paragraph(self, line: str, current_para: str,
                                idx: int, all_lines: List[str]) -> bool:


    def _detect_language(self, text: str) -> str:
        """Detect language using langid"""


    def _chunk_text(self, text: str) -> List[str]:
        """
        Smart chunking with clean boundaries:
        - Split by paragraphs first
        - Use sentence boundaries for large paragraphs
        - Never end chunks with commas or conjunctions
        """



    def _clean_chunk_ending(self, text: str) -> str:
        """Ensure chunk doesn't end badly"""


    def _split_sentences(self, text: str) -> List[str]:
        """Split text into sentences"""


    def translate_pdf(self, pdf_path: str) -> Dict:
        """Translate extracted chunks to English"""
        # returns sth like:
                    result = {
                "pdf_path": str(pdf_path),
                "language": lang,
                "translated": True,
                "chunk_pairs": [
                    {"original": orig, "translated": trans}
                    for orig, trans in zip(chunks, translated_chunks)
                ],
                "translated_at": str(datetime.now())
            }


    def _load_translator(self):
        """Load NLLB model once for all languages"""



    def _translate_chunks_nllb(self, chunks: List[str], src_lang: str) -> List[str]:
        """Fast batch translation using NLLB"""


    def inspect_extraction(self, pdf_path: str, n_samples=3):
        """Show extracted chunks with statistics"""


    def inspect_translation(self, pdf_path: str, n_samples=2):
        """Show original vs translated"""

    def load_model(self, model_name: str, task_name: str):
        """Load and cache a model"""

    def get_chunks_for_analysis(self, pdf_path: str, use_translated: bool = True) -> List[str]:
        """Get chunks ready for analysis"""


    def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter for climate-related chunks"""

    def process_pdfs(self, path: str, extract: bool = True,
                    translate: bool = True, skip_errors: bool = True):
        """
        Process PDF(s) - handles both single files and directories

        Args:
            path: Path to PDF file or directory (e.g., "data/report.pdf" or "data/reports/")
            extract: Whether to extract text
            translate: Whether to translate (requires extract=True)
            skip_errors: Continue on errors instead of stopping (only for directories)

        Returns:
            Dict with statistics and any errors encountered
        """

    def list_pdfs(self, root_dir: str, show_cached: bool = True):
        """
        List all PDFs and their cache status

        Args:
            root_dir: Directory to search
            show_cached: Show which PDFs already have cache files
        """



SyntaxError: incomplete input (2096783590.py, line 30)

### old

In [ ]:
def filter_climate_chunks(self, chunks: List[str], batch_size=None) -> List[Dict]:
        """Filter for climate-related chunks"""
        if batch_size is None:
            batch_size = self.BATCH_SIZE

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector",
            "detector"
        )

        print(f"\nFiltering {len(chunks)} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), batch_size), desc="Detecting"):
            batch = chunks[i:i+batch_size]

            try:
                results = detector(batch, truncation=True, max_length=512,
                                 batch_size=batch_size)

                for chunk, result in zip(batch, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'text': chunk,
                            'detector_score': result['score']
                        })
            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        print(f"✓ Kept {len(climate_chunks)} climate chunks "
              f"({len(climate_chunks)/len(chunks)*100:.1f}%)")
        return climate_chunks